<a href="https://colab.research.google.com/github/syankov-ai/Medium/blob/main/Agents/Birthday_notifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install faker

In [31]:
from dataclasses import dataclass, asdict
from datetime import date
from typing import List
from faker import Faker
import random
import pandas as pd

## Generate Data

In [32]:
fake = Faker()
Faker.seed(42)


@dataclass
class User:
    user_id: int
    name: str
    email: str
    birthday: date
    is_active: bool

def generate_fake_users(n: int = 10) -> List[User]:
    """Generate small user list with some birthdays equal to 'today'.

    Uses custom name-to-email mapping: "Denise Walker" -> "denise.walker@example.com"
    """
    users = []
    today = date.today()

    # force a couple of users to have birthday today
    birthday_today_indices = set(random.sample(range(n), k=min(2, n)))

    for i in range(n):
        # Generate name and create matching email (firstname.lastname@example.com)
        name = fake.name()
        first_last = name.lower().replace(' ', '.').replace('-', '.').replace("'", '')
        email = f"{first_last}@greatmail.com"

        if i in birthday_today_indices:
            # keep year realistic, but month/day = today
            year = random.randint(1970, 2005)
            birthday = date(year, today.month, today.day)
        else:
            # random birthday between 18 and 80 years old
            dob = fake.date_of_birth(minimum_age=18, maximum_age=80)
            birthday = dob

        user = User(
            user_id=i + 1,
            name=name,
            email=email,
            birthday=birthday,
            is_active=random.choice([True, True, False])
        )
        users.append(user)

    return users


## Birthday Job

In [33]:
def users_with_birthday_today(users: List[User]) -> List[User]:
    """Return users whose birthday is today (month/day match)."""
    today = date.today()
    return [
        u for u in users
        if u.is_active and u.birthday.month == today.month and u.birthday.day == today.day
    ]


def build_birthday_message(user: User) -> str:
    """Prompt-like message to be sent automatically, no extra confirmation."""
    return (
        f"🎂 Happy Birthday, {user.name}! "
        f"We’re so glad to have you with us. "
        f"Enjoy your day and watch out for a small surprise in your account. "
        f"— Your Customer Care Team"
    )


def send_birthday_message(user: User, dry_run: bool = True, is_first_message: bool = False) -> None:
    """
    In production this would call an email/SMS/Push provider.
    For now we just print the payload.
    """
    message = build_birthday_message(user)
    payload = {
        "to": user.email,
        "channel": "email",
        "template": "birthday_no_confirmation",
        "body": message,
        "metadata": {
            "user_id": user.user_id,
            "reason": "birthday",
            "auto_sent": True
        }
    }

    if dry_run:
        if is_first_message:
            # Article-friendly first message display
            print("\n" + "="*60)
            print("🎉 FIRST BIRTHDAY MESSAGE (ready to send):")
            print("="*60)
            print(f"📧 To: {user.name} <{user.email}>")
            print(f"📅 Birthday: {user.birthday}")
            print(f"💬 Message:\n{message}")
            print(f"📊 Metadata: {payload['metadata']}")
            print("="*60 + "\n")
        else:
            # Compact for additional messages
            print(f"[DRY-RUN] {user.name}: {message[:60]}...")
    else:
        # here you would integrate with an SMTP API or messaging provider
        pass


def run_daily_birthday_job() -> None:
    """
    Example of a 'daily job' that:
    1) loads users,
    2) finds birthdays,
    3) sends prompt-style messages without extra confirmation.
    """
    users = generate_fake_users(n=10)
    print("All users:")

    df_users = pd.DataFrame.from_records([asdict(u) for u in users])
    display(df_users.head(10))

    print("\nUsers with birthday today:")
    birthday_users = users_with_birthday_today(users)
    for u in birthday_users:
        print(asdict(u))

    print("\nSending messages (dry run):")
    for idx, u in enumerate(birthday_users):
        is_first = (idx == 0)
        send_birthday_message(u, dry_run=True, is_first_message=is_first)



if __name__ == "__main__":
    run_daily_birthday_job()


All users:


,user_id,name,email,birthday,is_active
0,1,Allison Hill,allison.hill@greatmail.com,1959-02-11,False
1,2,Megan Mcclain,megan.mcclain@greatmail.com,1950-07-12,True
2,3,Allen Robinson,allen.robinson@greatmail.com,1984-01-19,True
3,4,Alyssa Gonzalez,alyssa.gonzalez@greatmail.com,1985-12-28,False
4,5,Connie Lawrence,connie.lawrence@greatmail.com,1996-01-16,True
5,6,Robert Wolfe,robert.wolfe@greatmail.com,1966-06-27,True
6,7,Tyler Rogers,tyler.rogers@greatmail.com,1993-01-19,False
7,8,Brent Abbott,brent.abbott@greatmail.com,1983-01-31,True
8,9,Monica Herrera,monica.herrera@greatmail.com,2006-05-10,False
9,10,Juan Calderon,juan.calderon@greatmail.com,1984-01-07,False



Users with birthday today:
{'user_id': 3, 'name': 'Allen Robinson', 'email': 'allen.robinson@greatmail.com', 'birthday': datetime.date(1984, 1, 19), 'is_active': True}

Sending messages (dry run):

🎉 FIRST BIRTHDAY MESSAGE (ready to send):
📧 To: Allen Robinson <allen.robinson@greatmail.com>
📅 Birthday: 1984-01-19
💬 Message:
🎂 Happy Birthday, Allen Robinson! We’re so glad to have you with us. Enjoy your day and watch out for a small surprise in your account. — Your Customer Care Team
📊 Metadata: {'user_id': 3, 'reason': 'birthday', 'auto_sent': True}

